In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm
import scipy.stats as ss
from scipy.optimize import curve_fit
from scipy.stats.qmc import Halton

#Given Data
S0 = 35
s = 0.45
r = 0.03
q = 0.04
K = 35
T = 1



In [ ]:
#Task 1
#Black-Scholes Model, put call parity

def put_option (S0, K, T, r, q, s):
    d1 = (np.log(S0/K) + (r - q + 0.5 * s ** 2) * T) / (s * np.sqrt(T))
    d2 = d1 - s * np.sqrt(T)
    put_price = K * np.exp(-r * T) * norm.cdf(-d2) - S0 * np.exp(-q * T) * norm.cdf(-d1)
    return put_price

p = put_option(S0, K, T, r, q, s)
print(p)

def call_price(S0, K, T, r, q, s):
    d1 = (np.log(S0/K) + (r - q + 0.5 * s ** 2) * T) / (s * np.sqrt(T))
    d2 = d1 - s * np.sqrt(T)
    call_price = S0 * np.exp(-q * T) * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)
    return call_price

c = call_price(S0, K, T, r, q, s)
print(c)

def Put_Call_Parity(S0, K, T, r, q, put_price):
    call_price = S0 * np.exp(-q * T) - K * np.exp(-r * T) + put_price
    return call_price

c = Put_Call_Parity(S0, K, T, r, q, p)
print(c)

In [ ]:
#Task 2
#Bionimial Tree


def binomial_tree(S0, K, T, r, div, s, n):
    dt = T/n
    u = np.exp(s * np.sqrt(dt))
    d = 1/u
    p = (np.exp((r - div) * dt) - d) / (u - d)
    q = 1 - p
    stock_price = np.zeros((n+1, n+1))
    option_price = np.zeros((n+1, n+1))
    stock_price[0, 0] = S0
    for i in range(1, n+1):
        stock_price[i, 0] = stock_price[i-1, 0] * u
        for j in range(1, i+1):
            stock_price[i, j] = stock_price[i-1, j-1] * d
    for j in range(n+1):
        option_price[n, j] = np.maximum(0, K - stock_price[n, j])
    for i in range(n-1, -1, -1):
        for j in range(i+1):
            option_price[i, j] = np.exp(-r * dt) * (p * option_price[i+1, j] + q * option_price[i+1, j+1])
    return option_price[0, 0]

forty_steps = binomial_tree(S0, K, T, r, q, s, 40000)
print(forty_steps)

#illustrate
steps = [1,2,3,4,5,6,7,8,9,10,20,30,50,100,200,500,1000]
prices = []
for step in steps:
    prices.append(binomial_tree(S0, K, T, r, q, s, step))

plt.figure(figsize=(10, 6))
plt.plot(steps, prices, label='Option Price')
plt.xscale('log')
plt.xlabel('Number of Steps')
plt.ylabel('Option Price')
plt.title('Option Price vs Number of Steps')
plt.grid(True)
plt.show()

In [ ]:
#Task 3
#American floating lookback put option using binomial tree

def valuation(p, q, i, j, n, maxs, r, dt, stock_price):
    if i == n:  
        return max(0, maxs - stock_price[i, j])

    new_max = max(maxs, stock_price[i, j])

    option_value = np.exp(-r * dt) * (
        p * valuation(p, q, i + 1, j, n, new_max, r, dt, stock_price) +
        q * valuation(p, q, i + 1, j + 1, n, new_max, r, dt, stock_price)
    )

    
    exercise_value = max(0, new_max - stock_price[i, j])
    return max(option_value, exercise_value)


def american_floating_strike_put_option(S0, K, T, r, div, s, n):
    dt = T / n
    u = np.exp(s * np.sqrt(dt))
    d = 1 / u
    p = (np.exp((r - div) * dt) - d) / (u - d)
    q = 1 - p

   
    stock_price = np.zeros((n + 1, n + 1))
    stock_price[0, 0] = S0
    for i in range(1, n + 1):
        stock_price[i, 0] = stock_price[i - 1, 0] * u
        for j in range(1, i + 1):
            stock_price[i, j] = stock_price[i - 1, j - 1] * d

    
    option_price = valuation(p, q, 0, 0, n, S0, r, dt, stock_price)

    return option_price

prices = np.zeros(20)

for i in range(3,26):
    price = american_floating_strike_put_option(S0, K, T, r, q, s, i)
    print(price)
    prices[i] = price

In [ ]:
#Task 4
#Monte Carlo Simulation

dt=252

#4a

np.random.seed(seed = 3)

def monte_carlo(S0, T, r, q, s, n):
    h_sampler = Halton(d = 1, scramble  = True, seed = 7)
    h_samples = h_sampler.random(n)
    st_no_samples = ss.norm.ppf(h_samples)
    
    Prices = (r - q - 0.5 * s ** 2) * T + s * np.sqrt(T) * st_no_samples.flatten()
    Final_Price = S0 * np.exp(Prices)
    
    put_prices = np.exp(-r * T) * np.maximum(K - Final_Price, 0)
    s_e = np.std(put_prices) / np.sqrt(n)
    put_price = np.mean(put_prices)
    
    return put_price , s_e

N = 100000000

put_price, s_e = monte_carlo(S0, T, r, q, s, N)
print(put_price, s_e)

z_score = ss.norm.ppf(0.995)
conf_int = (put_price - z_score * s_e, put_price + z_score * s_e)
print(conf_int)

#4b

N = 1000000

def monte_carlo(S0, T, r, q, s, n):
    h_sampler = Halton(d = 1, scramble  = True, seed = 7)
    h_samples = h_sampler.random(n)
    
    st_no_samples = ss.norm.ppf(h_samples)

    Prices = (r - q - 0.5 * s ** 2) * T + s * np.sqrt(T) * st_no_samples.flatten()
    Final_Price = S0 * np.exp(Prices)

    return Final_Price

def simulate_interest_rate_paths(r0, alpha, beta, rho, steps, n):
    dt = 1 / steps
    #here we do another trick that is to generate two dimensions of halton values to reduce computing time
    h_sampler = Halton(d=2, scramble=True, seed=7)
    h_samples = h_sampler.random(n * steps).reshape(n, steps, 2)

    Z_stock = ss.norm.ppf(h_samples[:, :, 0])
    Z = ss.norm.ppf(h_samples[:, :, 1])

    Z_ir = rho * Z_stock + np.sqrt(1 - rho**2) * Z

    rates = np.zeros((n, steps + 1))
    rates[:, 0] = r0

    for i in range(1, steps + 1):
        rates[:, i] = rates[:, i - 1] * np.exp((alpha - 0.5 * beta**2) * dt + 
        beta * np.sqrt(dt) * Z_ir[:, i - 1]
        )

    return rates


rates = simulate_interest_rate_paths(0.01, 0.001, 0.05, 0.5, 252, N)

rT = rates[:, -1]

disc_r = np.exp(-np.sum(rates[:, 1:] * (T/dt), axis=1))


opt_values = disc_r * np.maximum(K - monte_carlo(S0, T, rT, q, s, N), 0)
put_price_2 = np.mean(opt_values)

s_e_2 = np.std(opt_values) / np.sqrt(N)
z_score = ss.norm.ppf(0.975)
conf_int_2 = (put_price_2 - z_score * s_e_2, put_price_2 + z_score * s_e_2)


print(put_price_2, s_e_2, conf_int_2)

#4c

#task 4c down and out put option with monte carlo

def down_and_out_put_option(S0,K, t, r, q, s, B, n):
    dt = 1/t
    value = np.zeros(n)
    S_paths = np.zeros((n, t+1))
    S_paths[:, 0] = S0
    
    h_sampler_ex = Halton(d = 1, scramble  = True, seed = 7)
    h_samples_ex = h_sampler_ex.random(n * t).reshape(n, t)
    Z_ex = ss.norm.ppf(h_samples_ex)

    #the catch here is that we create a matrix with values true and false(1 and 0) and if we hit the barrier then we set the value to false
    active_paths = np.ones(n, dtype=bool)


    for i in range(1, t + 1):
        S_paths[active_paths, i] = S_paths[active_paths, i - 1] * np.exp(
            (r - q - 0.5 * s ** 2) * dt + s * np.sqrt(dt) * Z_ex[active_paths, i - 1]
            )
        #if the stock price is below the barrier we set the path to false
        active_paths &= S_paths[:, i] >= B

    for i in range(n):
        if active_paths[i]:
            value[i] = np.maximum(K - S_paths[i, -1], 0)
    
    s_e = np.std(value) / np.sqrt(n)
    option_price = np.mean(value * np.exp(-r * T))

    return option_price, s_e





B = 30
N = 50000
t = 252


down_and_out_put_price, s_e_exotic = down_and_out_put_option(S0, K, t, r, q, s, B, N)
z_score = ss.norm.ppf(0.975)
conf_int_exotic = (down_and_out_put_price - z_score * s_e_exotic, down_and_out_put_price + z_score * s_e_exotic)

print(down_and_out_put_price,s_e_exotic, conf_int_exotic)

In [ ]:
#Task 7 Longstaff-Schwartz

np.random.seed(seed = 3)
n = 252
paths = 1000000
dt = T/n
df = np.exp(-r * dt)
degrees = 4
Sp = np.zeros((paths, 1))
W = ss.norm.rvs((r - q - 0.5 * s**2) * dt, np.sqrt(dt) * s, (paths, n - 1))
X = np.concatenate((Sp, W), axis=1).cumsum(1)

S = S0 * np.exp(X)


H = np.maximum(K - S, 0)  
V = np.zeros_like(H) 


V[:, -1] = H[:, -1]



for t in range(n - 2, 0, -1):
    good_paths = H[:, t] > 0 
    

    rg = np.polyfit(S[good_paths, t], V[good_paths, t + 1] * df, degrees)  
    C = np.polyval(rg, S[good_paths, t])  

    exercise = np.zeros(len(good_paths), dtype=bool)  
    exercise[good_paths] = H[good_paths, t] > C  

    V[exercise, t] = H[exercise, t]  
    V[exercise, t + 1 :] = 0  
    discount_path = V[:, t] == 0 
    V[discount_path, t] = V[discount_path, t + 1] * df  

V0 = np.mean(V[:, 1]) * df  
print(V0)